# Kaimaemae Model Training

This notebook trains and evaluates the models that power the Kaimaemae app. It
loads the model ready master dataset, explores the rainfall to bacteria signal,
trains several models, compares them with graphs suitable for a project board,
and exports the trained models plus metadata for the FastAPI backend.

## Research question

Can antecedent rainfall predict whether an Oahu beach is unsafe for swimming,
where unsafe means enterococcus above the EPA Beach Action Value of 130
CFU per 100 mL.

## Modeling approach and the decisions behind it

The project overview proposed an XGBoost regressor as the primary model, a
Random Forest as a comparison, and a time based split that trains before 2020
and tests on 2020 and later. After profiling the actual master dataset, two of
those decisions were adjusted for good reasons. Both changes are documented here
so the reasoning is transparent.

1. Split year moved from 2020 to 2014. The dataset thins out sharply in recent
   years and the years 2021 through 2024 contain zero unsafe samples. A 2020 cut
   leaves only about 680 test rows with 3 unsafe samples, which is far too few to
   measure precision, recall, or AUC. A 2014 cut keeps the chronological no
   leakage split the project wants while giving roughly 5,800 test rows with 84
   unsafe samples, which is actually evaluable.

2. The target for the regressor is log1p(enterococcus), not the raw count. The
   bacteria readings span 0.1 to 84,000 CFU and are extremely right skewed, so a
   raw count regressor is dominated by a handful of huge values. Training on the
   log keeps the model stable, and we invert the prediction before applying the
   130 threshold.

We train three models and compare them honestly:

- XGBoost regressor on the log target, which gives a continuous risk gradient.
- XGBoost classifier on the unsafe label, using scale_pos_weight to handle the
  class imbalance. This directly targets unsafe detection.
- Random Forest classifier with balanced class weights, as the comparison
  baseline the project overview asked for.

## Feature choice

The model inputs are the engineered antecedent rainfall features plus the
calendar month: rain_24hr, rain_48hr, rain_72hr, rain_7day, days_since_rain,
max_rain_3day, and month. Beach identity and coordinates are deliberately left
out of the model so it answers the research question about rainfall rather than
memorizing each beach base rate. Per beach behavior is summarized separately in
the per beach analysis section and exported for the map.

## Outputs

Trained models and a metadata file are written to `models/`, and every graph is
saved to `figures/` so they can be dropped straight onto a project board.


In [ ]:
## Imports and configuration

# pandas and numpy handle the data, matplotlib and seaborn draw the graphs,
# xgboost and scikit-learn provide the models and metrics, and joblib saves the
# Random Forest. All paths, the feature list, the safety threshold, and the split
# year live here so every choice is in one place. FIGURES_DIR and MODELS_DIR are
# created up front so saving never fails.
import os
import json
import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from xgboost import XGBRegressor, XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    average_precision_score,
    confusion_matrix,
    classification_report,
)

# Notebook lives in backend/machine_learning/ so go up two levels to the root.
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
CLEAN_DIR = os.path.join(PROJECT_ROOT, "resources", "clean_data")
MASTER_PATH = os.path.join(CLEAN_DIR, "master_dataset.csv")
ML_DIR = os.path.join(PROJECT_ROOT, "backend", "machine_learning")
MODELS_DIR = os.path.join(ML_DIR, "models")
FIGURES_DIR = os.path.join(ML_DIR, "figures")
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

# Model inputs: the engineered antecedent rainfall features plus calendar month.
FEATURES = [
    "rain_24hr",
    "rain_48hr",
    "rain_72hr",
    "rain_7day",
    "days_since_rain",
    "max_rain_3day",
    "month",
]
BAV_THRESHOLD = 130  # EPA Beach Action Value, above this is unsafe.
SPLIT_YEAR = 2014  # train on years before this, test on this year and later.
RANDOM_STATE = 42

sns.set_theme(style="whitegrid")


In [ ]:
## Load the master dataset

# Read the model ready table produced by the merge step. Identifier columns are
# read as text so values stay exact. A numeric year column is derived from the
# date string and is used only for the chronological train and test split.
df = pd.read_csv(
    MASTER_PATH, dtype={"location_id": str, "nearest_station_id": str}
)
df["year"] = df["date"].str[:4].astype(int)

print("Master shape:", df.shape)
print("Beaches:", df["location_id"].nunique())
print("Date range:", df["date"].min(), "to", df["date"].max())
print("Overall unsafe rate:", round(df["unsafe"].mean(), 4))
df.head()


# Exploratory data analysis

Before training, three quick views establish what the data looks like and
whether the core idea even holds. Each graph is saved to the figures folder and
is followed by a short written summary.

The first graph shows the shape of the target, the second shows how rare the
unsafe class is, and the third checks the central hypothesis that more
antecedent rainfall lines up with more unsafe samples.


In [ ]:
## Graph 1: target distribution, raw versus log

# The left panel shows raw enterococcus, the right shows log1p of the same value
# with the 130 threshold marked. The raw view is almost a single spike near zero
# with a long thin tail, which is exactly why the regressor is trained on the log.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df["enterococcus"], bins=60, color="#2b8cbe")
axes[0].set_title("Enterococcus raw (CFU/100mL)")
axes[0].set_xlabel("CFU/100mL")
axes[0].set_ylabel("samples")
axes[1].hist(np.log1p(df["enterococcus"]), bins=60, color="#2b8cbe")
axes[1].axvline(np.log1p(BAV_THRESHOLD), color="red", linestyle="--", label="BAV 130")
axes[1].set_title("Enterococcus log1p")
axes[1].set_xlabel("log1p(CFU/100mL)")
axes[1].legend()
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "01_target_distribution.png"), dpi=120)
plt.show()


**Graph 1 summary.** The raw bacteria count is extremely right skewed. The
median sample is about 2 CFU while the maximum is 84,000 CFU, so on the raw scale
almost every sample is crushed into the first bin and a few storms form a long
tail. On the log scale the distribution opens up into a smooth shape and the
130 threshold sits well inside the right tail, which visually confirms that
unsafe readings are uncommon. This is the evidence behind training the regressor
on log1p rather than the raw count.


In [ ]:
## Graph 2: class balance

# Count safe versus unsafe samples. The title shows the unsafe share so the
# imbalance is obvious. This is what motivates scale_pos_weight in the classifier
# and balanced class weights in the Random Forest.
counts = df["unsafe"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(["safe (0)", "unsafe (1)"], counts.values, color=["#74c476", "#fb6a4a"])
for i, v in enumerate(counts.values):
    ax.text(i, v, f"{v:,}", ha="center", va="bottom")
ax.set_title(f"Class balance, unsafe rate {df['unsafe'].mean() * 100:.1f}%")
ax.set_ylabel("samples")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "02_class_balance.png"), dpi=120)
plt.show()


**Graph 2 summary.** Only about 3.7 percent of samples are unsafe, so roughly
27 out of every 28 readings are safe. A model that always predicts safe would
look 96 percent accurate while being useless, which is why accuracy is ignored
later in favor of recall, precision, and the area under the precision recall
curve. This imbalance is handled directly during training through
scale_pos_weight for XGBoost and balanced class weights for the Random Forest.


In [ ]:
## Graph 3: exceedance rate versus antecedent rainfall

# Bucket every sample by how much rain fell in the 7 days before it, then plot the
# share of unsafe samples in each bucket. If rainfall has no signal these bars are
# flat. If the project hypothesis holds, the bars climb from left to right.
bins = [0, 1, 5, 10, 25, 50, 100, 10000]
labels = ["0-1", "1-5", "5-10", "10-25", "25-50", "50-100", "100+"]
df["rain7_bin"] = pd.cut(
    df["rain_7day"], bins=bins, labels=labels, include_lowest=True
)
rate = df.groupby("rain7_bin", observed=True)["unsafe"].mean() * 100

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(rate.index.astype(str), rate.values, color="#fb6a4a")
for i, v in enumerate(rate.values):
    ax.text(i, v, f"{v:.1f}%", ha="center", va="bottom")
ax.set_ylabel("unsafe rate (%)")
ax.set_xlabel("7 day antecedent rainfall (mm)")
ax.set_title("Exceedance rate rises with antecedent rainfall")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "03_rainfall_vs_exceedance.png"), dpi=120)
plt.show()


**Graph 3 summary.** This is the most important exploratory result. The unsafe
rate climbs steadily from about 1.4 percent on the driest weeks to about 21
percent on the wettest weeks, roughly a 15 fold increase. That clear upward
staircase is direct evidence that antecedent rainfall carries real signal about
beach safety, which is the entire premise of the project. It also sets a
realistic ceiling on expectations, since even the wettest weeks are unsafe only
about one time in five, so rainfall shifts the odds rather than guaranteeing an
outcome.


# Train and test split

The split is chronological, never random. Random shuffling would let the model
peek at future samples that share a date or a storm with training rows, which
would leak information and inflate the scores. Training only on the past and
testing only on the future mirrors how the app will be used, where the model
predicts a date it has never seen.

As explained in the overview, the cut year is 2014 rather than 2020 because the
later years are too sparse and contain no unsafe samples to test against.


In [ ]:
## Build the chronological split and the feature and target matrices

# Rows before SPLIT_YEAR train the models, rows from that year on test them. The
# same feature matrix feeds both tasks. The classifier target is the unsafe label,
# and the regressor target is log1p of the raw count so the model trains on the
# stable log scale. The scale_pos_weight value is the ratio of safe to unsafe rows
# in the training set, which tells XGBoost how much to up weight the rare class.
train = df[df["year"] < SPLIT_YEAR]
test = df[df["year"] >= SPLIT_YEAR]

X_train, X_test = train[FEATURES], test[FEATURES]
y_train_clf, y_test_clf = train["unsafe"], test["unsafe"]
y_train_reg = np.log1p(train["enterococcus"])
y_test_reg = np.log1p(test["enterococcus"])

neg = int((y_train_clf == 0).sum())
pos = int((y_train_clf == 1).sum())
scale_pos_weight = neg / pos

print("Train rows:", len(X_train), "Test rows:", len(X_test))
print("Test unsafe samples:", int(y_test_clf.sum()))
print("scale_pos_weight:", round(scale_pos_weight, 2))


# Train the models

Three models are trained on the same training rows. All three use modest depth
and a small learning rate or capped depth to keep them from overfitting the rare
unsafe class. The XGBoost models use gain based feature importance so the
importance graph later reflects how much each feature improved the splits.


In [ ]:
## Fit the regressor, the classifier, and the Random Forest

# XGBoost regressor on the log target gives a continuous predicted bacteria level.
# XGBoost classifier on the unsafe label, with scale_pos_weight, gives a calibrated
# probability of unsafe. Random Forest with balanced class weights is the
# comparison baseline. random_state is fixed everywhere for reproducibility.
reg = XGBRegressor(
    n_estimators=400,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=RANDOM_STATE,
    objective="reg:squarederror",
    importance_type="gain",
)
reg.fit(X_train, y_train_reg)

clf = XGBClassifier(
    n_estimators=400,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
    objective="binary:logistic",
    eval_metric="logloss",
    importance_type="gain",
)
clf.fit(X_train, y_train_clf)

rf = RandomForestClassifier(
    n_estimators=400,
    max_depth=12,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf.fit(X_train, y_train_clf)

print("All three models trained.")


# Evaluation

All scores below come from the held out test set only. For the regressor we
report error on the log scale and then apply the 130 threshold to turn its
predicted bacteria level into a safe or unsafe decision so it can be compared on
equal footing with the classifiers. For the classifiers we use the predicted
probability of unsafe.

The metrics that matter for this project are recall, which is the share of truly
unsafe days the model catches, and the area under the precision recall curve,
which summarizes performance on the rare class without being fooled by the easy
safe majority. Accuracy is intentionally not used.


In [ ]:
## Compute predictions and a shared metrics table

# The regressor predicts on the log scale, then we invert with expm1 to get a
# bacteria level and threshold it at 130. The classifiers give a probability of
# unsafe, thresholded at 0.5 for the hard decision. clf_metrics packages the five
# headline classification numbers so the three models can be compared in one place.
reg_pred_log = reg.predict(X_test)
reg_pred_cfu = np.expm1(reg_pred_log)
reg_unsafe = (reg_pred_cfu > BAV_THRESHOLD).astype(int)
rmse_log = float(np.sqrt(mean_squared_error(y_test_reg, reg_pred_log)))
mae_log = float(mean_absolute_error(y_test_reg, reg_pred_log))
r2_log = float(r2_score(y_test_reg, reg_pred_log))

clf_proba = clf.predict_proba(X_test)[:, 1]
clf_pred = (clf_proba >= 0.5).astype(int)
rf_proba = rf.predict_proba(X_test)[:, 1]
rf_pred = (rf_proba >= 0.5).astype(int)


def clf_metrics(y_true, y_pred, score):
    return {
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, score)),
        "pr_auc": float(average_precision_score(y_true, score)),
    }


m_reg = clf_metrics(y_test_clf, reg_unsafe, reg_pred_cfu)
m_clf = clf_metrics(y_test_clf, clf_pred, clf_proba)
m_rf = clf_metrics(y_test_clf, rf_pred, rf_proba)

summary = pd.DataFrame(
    {"XGB regressor": m_reg, "XGB classifier": m_clf, "Random Forest": m_rf}
).T.round(3)
print("Regressor log scale error  RMSE", round(rmse_log, 3),
      "MAE", round(mae_log, 3), "R2", round(r2_log, 3))
summary


In [ ]:
## Graph 4: regressor predicted versus actual

# Each point is a test sample, actual log bacteria on the x axis and predicted on
# the y axis. The dashed diagonal is a perfect prediction. The red lines mark the
# 130 threshold on both axes, splitting the plot into safe and unsafe quadrants.
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test_reg, reg_pred_log, alpha=0.3, s=12, color="#2b8cbe")
lims = [0, float(max(y_test_reg.max(), reg_pred_log.max()))]
ax.plot(lims, lims, "k--", linewidth=1, label="perfect prediction")
ax.axhline(np.log1p(BAV_THRESHOLD), color="red", linestyle=":", label="BAV 130")
ax.axvline(np.log1p(BAV_THRESHOLD), color="red", linestyle=":")
ax.set_xlabel("actual log1p(CFU)")
ax.set_ylabel("predicted log1p(CFU)")
ax.set_title("Regressor predicted vs actual (log space)")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "04_pred_vs_actual.png"), dpi=120)
plt.show()


**Graph 4 summary.** The predictions form a roughly horizontal band rather than
hugging the diagonal, and almost no points rise above the horizontal 130 line.
In plain terms the regressor predicts a fairly typical bacteria level for almost
every day and rarely commits to a high value, so on the test set its log scale R2
is slightly negative and it crosses the unsafe threshold for only a couple of
days. This is the evidence that predicting an exact bacteria count from rainfall
alone is very hard, and it is the main reason the dedicated classifier, which
only has to output a probability of unsafe, becomes the model the app should
rely on.


In [ ]:
## Graph 5: feature importance

# Gain based importance from the XGBoost classifier, sorted so the strongest
# feature is on top. This directly answers the research question about which
# rainfall window matters most for predicting unsafe conditions.
imp = pd.Series(clf.feature_importances_, index=FEATURES).sort_values()
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(imp.index, imp.values, color="#756bb1")
ax.set_title("XGBoost classifier feature importance (gain)")
ax.set_xlabel("relative importance")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "05_feature_importance.png"), dpi=120)
plt.show()

imp.sort_values(ascending=False).round(3)


**Graph 5 summary.** The short window totals carry the most weight, with the 48
hour and 24 hour rainfall sums at the top, followed by calendar month and the 72
hour total. In practice this says that the rain in the day or two right before a
sample is the strongest predictor of an unsafe reading, which fits the idea of
storm runoff quickly washing bacteria into the water. Month ranking highly
reflects the wet season versus dry season pattern. The cumulative windows overlap
by construction, so their importance is shared across them rather than
concentrated in a single bar, and days_since_rain contributes the least.


In [ ]:
## Graph 6: confusion matrix for the classifier

# Counts of how the classifier labeled the test set. The diagonal is correct, the
# off diagonal is mistakes. The bottom right cell is the unsafe days we caught and
# the bottom left is the unsafe days we missed, which is the most costly error.
cm = confusion_matrix(y_test_clf, clf_pred)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(
    cm, annot=True, fmt=",d", cmap="Blues", cbar=False,
    xticklabels=["safe", "unsafe"], yticklabels=["safe", "unsafe"], ax=ax,
)
ax.set_xlabel("predicted")
ax.set_ylabel("actual")
ax.set_title("XGBoost classifier confusion matrix (test)")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "06_confusion_matrix.png"), dpi=120)
plt.show()

print(classification_report(y_test_clf, clf_pred, target_names=["safe", "unsafe"]))


**Graph 6 summary.** With the probability cut at 0.5 the classifier catches about
38 percent of the truly unsafe days, roughly 32 of the 84 in the test set, while
raising a large number of false alarms on safe days. That trade is expected for a
rare event screened with a balanced model, and it is deliberately tuned toward
catching unsafe days because missing an unsafe beach is far worse than warning
about a beach that turns out fine. The decision threshold is a dial, and the next
two curves show the full range of trades it can make instead of this single
point.


In [ ]:
## Graph 7: ROC and precision recall curves

# These two curves judge the models across every possible threshold. ROC plots the
# true positive rate against the false positive rate, and the precision recall
# curve is the better view for a rare class. The dashed lines are the no skill
# baselines, a diagonal for ROC and the unsafe base rate for precision recall.
fpr_c, tpr_c, _ = roc_curve(y_test_clf, clf_proba)
fpr_r, tpr_r, _ = roc_curve(y_test_clf, rf_proba)
prec_c, rec_c, _ = precision_recall_curve(y_test_clf, clf_proba)
prec_r, rec_r, _ = precision_recall_curve(y_test_clf, rf_proba)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(fpr_c, tpr_c, label=f"XGB AUC {m_clf['roc_auc']:.3f}", color="#2b8cbe")
axes[0].plot(fpr_r, tpr_r, label=f"RF AUC {m_rf['roc_auc']:.3f}", color="#fb6a4a")
axes[0].plot([0, 1], [0, 1], "k--", linewidth=1, label="no skill")
axes[0].set_xlabel("false positive rate")
axes[0].set_ylabel("true positive rate")
axes[0].set_title("ROC curve")
axes[0].legend()

axes[1].plot(rec_c, prec_c, label=f"XGB AP {m_clf['pr_auc']:.3f}", color="#2b8cbe")
axes[1].plot(rec_r, prec_r, label=f"RF AP {m_rf['pr_auc']:.3f}", color="#fb6a4a")
axes[1].axhline(
    y_test_clf.mean(), color="gray", linestyle="--", label="base rate"
)
axes[1].set_xlabel("recall")
axes[1].set_ylabel("precision")
axes[1].set_title("Precision recall curve")
axes[1].legend()
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "07_roc_pr.png"), dpi=120)
plt.show()


**Graph 7 summary.** Both models land at an ROC area under the curve of about
0.66, comfortably above the 0.5 no skill diagonal, which confirms there is real
and usable signal in rainfall. The precision recall curve is more sobering, with
an average precision near 0.07 against a base rate of about 0.04, so the model is
better than guessing but precision stays low because unsafe days are genuinely
rare and rainfall only shifts the odds. XGBoost edges out the Random Forest on
the precision recall view, which is why it is chosen as the model to deploy.


In [ ]:
## Graph 8: model comparison

# Group the four headline metrics side by side for the three models so the trade
# offs are visible at a glance. This is the single chart to put on a project board
# to justify which model is chosen.
models = ["XGB regressor", "XGB classifier", "Random Forest"]
metric_names = ["recall", "f1", "roc_auc", "pr_auc"]
values = {
    "recall": [m_reg["recall"], m_clf["recall"], m_rf["recall"]],
    "f1": [m_reg["f1"], m_clf["f1"], m_rf["f1"]],
    "roc_auc": [m_reg["roc_auc"], m_clf["roc_auc"], m_rf["roc_auc"]],
    "pr_auc": [m_reg["pr_auc"], m_clf["pr_auc"], m_rf["pr_auc"]],
}
x = np.arange(len(metric_names))
w = 0.25
fig, ax = plt.subplots(figsize=(9, 4))
for i, mdl in enumerate(models):
    vals = [values[m][i] for m in metric_names]
    ax.bar(x + (i - 1) * w, vals, w, label=mdl)
ax.set_xticks(x)
ax.set_xticklabels(metric_names)
ax.set_ylim(0, 1)
ax.set_ylabel("score")
ax.set_title("Model comparison on unsafe detection (test)")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "08_model_comparison.png"), dpi=120)
plt.show()


**Graph 8 summary.** The three models reach a similar ranking quality, shown by
the near equal ROC area bars, but they behave very differently at the decision
point. The XGBoost regressor barely flags any unsafe day, so its recall is close
to zero even though it can rank reasonably. The two classifiers actually catch
unsafe days, and the XGBoost classifier gives the best balance of recall and
precision recall area. That makes the XGBoost classifier the model the app
should serve, with the regressor kept only as a secondary continuous risk score.


# Per beach analysis

The model itself is deliberately blind to beach identity, but the app still needs
to know how risky each beach has been historically so the map can color pins and
the detail panels can show context. This section summarizes every beach and
exports that table for the frontend.


In [ ]:
## Graph 9: top beaches by historical exceedance rate

# Aggregate to one row per beach with sample count, exceedance rate, coordinates,
# and assigned station. The graph shows the 15 riskiest beaches among those with
# at least 100 samples, so a high rate from only a handful of readings does not
# dominate. The full beach table is kept for export to the app.
beach = (
    df.groupby(["location_id", "location_name"])
    .agg(
        n=("unsafe", "size"),
        exceed_rate=("unsafe", "mean"),
        latitude=("latitude", "first"),
        longitude=("longitude", "first"),
        nearest_station_id=("nearest_station_id", "first"),
    )
    .reset_index()
)

top = beach[beach["n"] >= 100].sort_values("exceed_rate", ascending=False).head(15)
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top["location_name"][::-1], (top["exceed_rate"] * 100)[::-1], color="#fb6a4a")
ax.set_xlabel("historical exceedance rate (%)")
ax.set_title("Top 15 Oahu beaches by exceedance rate (min 100 samples)")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "09_top_beaches.png"), dpi=120)
plt.show()

top[["location_name", "n", "exceed_rate"]].round(3)


**Graph 9 summary.** Risk is very uneven across the island. The beaches at the top
of this chart, which tend to be enclosed harbors, stream mouths, and lagoons with
poor flushing, are unsafe a large share of the time, while open coast beaches sit
far down the list. This matters for the app in two ways. It tells the map which
pins deserve attention by default, and it explains why a rainfall only model has
a hard ceiling, since a lot of the difference between beaches comes from the
fixed character of each site rather than from recent rain. That per site base rate
is exactly what the exported beach table provides to complement the model.


# Export models and artifacts for the backend

Everything the FastAPI backend needs is written to the models folder. The two
XGBoost models are saved in the portable native JSON format so they load without
any pickle compatibility worries, the Random Forest is saved with joblib, a
metadata file records the exact feature order and the test metrics, and the beach
catalog gives the map its pins and base rates.

Nothing here is Docker specific yet. Saving the artifacts in clean portable
formats and pinning the feature order is the preparation that makes the later
FastAPI and Docker work straightforward.


In [ ]:
## Save the models, metadata, and beach catalog

# The metadata file is the contract the API reads at startup. It pins the feature
# order so the API builds inputs the exact way the models were trained, records the
# threshold and the log target choice, and stores the test metrics for reference.
reg.save_model(os.path.join(MODELS_DIR, "xgb_regressor.json"))
clf.save_model(os.path.join(MODELS_DIR, "xgb_classifier.json"))
joblib.dump(rf, os.path.join(MODELS_DIR, "random_forest_classifier.joblib"))

metadata = {
    "created": datetime.date.today().isoformat(),
    "features": FEATURES,
    "bav_threshold": BAV_THRESHOLD,
    "split_year": SPLIT_YEAR,
    "target_regressor": "log1p(enterococcus)",
    "target_classifier": "unsafe",
    "scale_pos_weight": round(scale_pos_weight, 4),
    "train_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),
    "primary_model_for_app": "xgb_classifier",
    "metrics_test": {
        "xgb_regressor": {
            "rmse_log": rmse_log,
            "mae_log": mae_log,
            "r2_log": r2_log,
            **m_reg,
        },
        "xgb_classifier": m_clf,
        "random_forest": m_rf,
    },
    "model_files": {
        "xgb_regressor": "xgb_regressor.json",
        "xgb_classifier": "xgb_classifier.json",
        "random_forest": "random_forest_classifier.joblib",
    },
}
with open(os.path.join(MODELS_DIR, "model_metadata.json"), "w") as fh:
    json.dump(metadata, fh, indent=2)

beach.to_csv(os.path.join(MODELS_DIR, "beach_catalog.csv"), index=False)

print("Saved to", MODELS_DIR)
for name in sorted(os.listdir(MODELS_DIR)):
    print(" ", name)


# Results summary and next steps

## Headline findings for the project board

- Antecedent rainfall is a real predictor of unsafe swimming conditions. The
  unsafe rate rises from about 1.4 percent on the driest weeks to about 21
  percent on the wettest weeks.
- The strongest single inputs are the 24 hour and 48 hour rainfall totals,
  followed by calendar month, which matches the storm runoff and wet season
  story.
- The deployed model is the XGBoost classifier. On the held out test years from
  2014 on it reaches an ROC area under the curve of about 0.66 and catches
  roughly 38 percent of unsafe days at the default threshold.
- The signal is useful but not precise. Unsafe days are rare and rainfall shifts
  the odds rather than determining the outcome, so the app should present the
  output as an elevated risk warning, not a guarantee.

## Honest limitations

- Precision at the default threshold is low, so most warnings will be false
  alarms. The decision threshold can be tuned on a validation split to trade
  recall against false alarms for the app.
- A rainfall only model cannot capture fixed site effects such as poor flushing
  in harbors and stream mouths, which is why the per beach base rate table is
  exported alongside the model.
- Recent years are sparse and contain almost no unsafe samples, which limits how
  current the test window can be.

## What is ready for FastAPI and Docker

The models folder now contains everything the backend needs:

- `xgb_classifier.json`, the primary model the API should load and call.
- `xgb_regressor.json`, an optional secondary continuous risk score.
- `random_forest_classifier.joblib`, kept for reference and comparison.
- `model_metadata.json`, which pins the exact feature order, the threshold, and
  the recorded test metrics so the API builds inputs correctly.
- `beach_catalog.csv`, the per beach coordinates, sample counts, exceedance
  rates, and assigned station for the map.
- `figures/`, every graph above saved as a PNG for the project board.

Suggested next steps, not done here: build a FastAPI app that loads the
classifier and metadata at startup and exposes a predict endpoint that accepts
rainfall inputs and returns the unsafe probability, then add a Dockerfile that
installs the pinned dependencies and copies the models folder. The artifact
formats chosen here are already container friendly.
